In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hammadfarooq470/google-play-store-most-downloaded-android-apps")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 기본 테마 설정
sns.set_theme(palette="viridis", style="darkgrid", font_scale=1)
sns.color_palette("viridis", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Umdot 12'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
playstore_df = pd.read_csv(f'{path}/google_play_store_most_downloaded_apps.csv') 

In [ ]:
playstore_df

# 정보 확인
- 그래요... 전처리하다 피똥싸게 생겼습니다... 

## .info()

In [ ]:
playstore_df.info()

## .describe()
- 다 오브젝트라 include='O'만 진행합니다. 

In [ ]:
playstore_df.describe(include='O')

## .isna().sum()

In [ ]:
playstore_df.isna().sum()

## .head()

In [ ]:
playstore_df.head()

## .colummns

In [ ]:
playstore_df.columns

# 전처뤼이이이이이아

## 가격 정상화
1. 가격의 숫자화
2. 무료앱 가격 땜질 

In [ ]:
# 일단 저 달러부터 떼보시죠 
playstore_df['Price']

### 달러 고 홈

In [ ]:
playstore_df['Price'] = playstore_df['Price'].str.replace('$', '', regex=False)
playstore_df['Price']

### 결측값 때우기

In [ ]:
# 일단 가격이 결측값이 0인 모든 앱의 가격들이 다 무료인지 봅시다. 
free_index = playstore_df.query('Price.isna()').index

for idx in free_index:
    print(playstore_df['Type'].loc[idx])

In [ ]:
playstore_df['Price'] = playstore_df['Price'].fillna(0) # 채우고 
playstore_df['Price'] = pd.to_numeric(playstore_df['Price']) # 바꾸면

playstore_df['Price'] # 정상화 끝 

## 날짜 정상화

In [ ]:
# 날짜가 두개지요? 
# 난 저 리치드가 제출일인 줄 알았는데 일고보니 특정 다운로드 수에 도달한 날짜였음... 그래서 리치드가 퍼블리시드보다 뒤 시점입니다. 
playstore_df['Date_Reached'] = pd.to_datetime(playstore_df['Date_Reached'], errors='coerce')
playstore_df['Date_Published'] = pd.to_datetime(playstore_df['Date_Published'], errors='coerce')

In [ ]:
playstore_df['Date_Reached']

- 아니 언노운때문에 오류뜨는거였냐고... 

### 년도 추출
- 발매년도 추출해봅시다. 

In [ ]:
# 발매년도
playstore_df['Release_year'] = playstore_df['Date_Published'].dt.year
playstore_df['Release_year']

In [ ]:
playstore_df

### 마일스톤 도달까지 걸린 일수

In [ ]:
# 이걸로 되는겨? 
diff = playstore_df['Date_Reached'] - playstore_df['Date_Published']

In [ ]:
playstore_df['To_reach'] = diff.dt.days

### 설마 도달일수에 문제 있는 데이터는 없겠지? 

In [ ]:
playstore_df.query('To_reach < 0')

- 응 있어 

In [ ]:
# 날립시다. 
playstore_df = playstore_df.dropna()
playstore_df

# 분석 렛츄고

## 마일스톤과 도달일

### 그룹별로 빨리 도달한 앱

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
min_idx = playstore_df.groupby('Downloads')['To_reach'].idxmin()

# 오케이 렛츠씨 
playstore_df.loc[min_idx]

- 게임이 두개나 보이는데 내가 모르는 게임임... 

### 그룹별로 오래걸린 앱

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
max_idx = playstore_df.groupby('Downloads')['To_reach'].idxmax()

# 오케이 렛츠씨 
playstore_df.loc[max_idx]

### 게임! 게임을 보자! 

In [ ]:
game_df = playstore_df.query('Category.str.contains("Game")')
game_df

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
min_idx = game_df.groupby('Downloads')['To_reach'].idxmin()

# 오케이 렛츠씨 
game_df.loc[min_idx]

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
max_idx = game_df.groupby('Downloads')['To_reach'].idxmax()

# 오케이 렛츠씨 
game_df.loc[max_idx]

### 앱 가격별 분석-무료

In [ ]:
free_df = playstore_df.query('Price == 0')
free_df

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
min_idx = free_df.groupby('Downloads')['To_reach'].idxmin()

# 오케이 렛츠씨 
free_df.loc[min_idx]

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
max_idx = free_df.groupby('Downloads')['To_reach'].idxmax()

# 오케이 렛츠씨 
free_df.loc[max_idx]

### 앱 가격별 분석-유료 앱

In [ ]:
paid_df = playstore_df.query('Price > 0')
paid_df

In [ ]:
# 마일스톤별로 묶은 다음 최솟값의 인덱스를 추출 
min_idx = paid_df.groupby('Downloads')['To_reach'].idxmin()

# 오케이 렛츠씨 
paid_df.loc[min_idx]

In [ ]:
# 마일스톤별로 묶은 다음 최댓값의 인덱스를 추출 
max_idx = paid_df.groupby('Downloads')['To_reach'].idxmax()

# 오케이 렛츠씨 
paid_df.loc[max_idx]

## 발매년도별로 보기

### 2010~2020
- 2010년 이전... 왜 있는지 모르겠습니다. 제가 갤럭시 A를 2010년애 봤음. 
- 그리고 애초에 하나밖에 없어요 2010년 이전에.

In [ ]:
year_min_idx = playstore_df.query('2010 <= Release_year < 2021').groupby(['Release_year'])['To_reach'].idxmin()
playstore_df.loc[year_min_idx]

In [ ]:
year_max_idx = playstore_df.query('2010 <= Release_year < 2021').groupby(['Release_year'])['To_reach'].idxmax()
playstore_df.loc[year_max_idx]

### 2021~

In [ ]:
year_min_idx = playstore_df.query('2021 <= Release_year').groupby(['Release_year'])['To_reach'].idxmin()
playstore_df.loc[year_min_idx]

In [ ]:
year_max_idx = playstore_df.query('2021 <= Release_year').groupby(['Release_year'])['To_reach'].idxmax()
playstore_df.loc[year_max_idx]

## 모스트 다은로드 오브 구글 앱

In [ ]:
google_df = playstore_df.query('Developer.str.contains("Google") and Downloads == "10B+"')
google_df

In [ ]:
google_df_s = google_df.sort_values('To_reach', ascending=False)

In [ ]:
sns.barplot(google_df_s, x = 'App', y = 'To_reach', hue = 'App', palette='viridis')
plt.title('To reach for milestone: Google apps')
plt.xlabel('App')
plt.xticks(rotation = 45)
plt.ylabel('To Reach (days)')
plt.show()

## 모스트 다운로드 오브 게임

In [ ]:
most_game_df = playstore_df.query('Category.str.contains("Games") and Downloads.str.contains("B")')
most_game_df = most_game_df.sort_values(['Downloads','To_reach'], ascending = [True, False])

In [ ]:
sns.barplot(most_game_df, x = 'App', y = 'To_reach', hue = 'Downloads', palette='viridis')
plt.title('To reach for milestone: Games')
plt.xlabel('App')
plt.xticks(rotation = 45)
plt.ylabel('To Reach (days)')
plt.show()